In [1]:
import numpy as np
import pandas as pd
import boto3
import s3fs
import io
import os
import random
import tempfile
import re
from tqdm import tqdm
from pathlib import Path

In [2]:
import wfdb
import ast

In [3]:
# Define the bucket name and prefixes 
bucket_name = 'walkky-datasets'
datasets = ['ptbxl', 'cpsc_2018', 'csn']

base_prefixes: dict[str, str] = {dataset:  f'data/{dataset}' for dataset in datasets}

# Output key
bucket_out = 'walkky-ml'
prefix_out = "aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning"

# Initialize the S3 client
s3_client = boto3.client('s3')
s3_fs = s3fs.S3FileSystem()


In [4]:
base_prefixes

{'ptbxl': 'data/ptbxl', 'cpsc_2018': 'data/cpsc_2018', 'csn': 'data/csn'}

In [5]:
# Reading csv files

def read_csv_from_s3(bucket_name, key, index_col):
    response = s3_client.get_object(Bucket=bucket_name, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), index_col=index_col)

In [6]:
# Writing the results as npz files to s3

def _signals_npz_key(dataset_name: str) -> str:
    return f"{prefix_out}/{dataset_name}_signals.npz"


def write_ecg_signals_to_s3(
    signals: list[np.ndarray],
    labels: np.ndarray,
    ids: np.ndarray,
    dataset_name: str,
    strat_fold: np.ndarray | None = None,
) -> None:
    """
    Write variable-length ECG signals + metadata to a single NPZ on S3.

    Parameters
    ----------
    signals    : list of (length_i, 12) float arrays — length_i may vary per record,
                 typically ~5000 but not guaranteed
    labels     : (N,) int array of labels, same order as `signals`
    ids        : (N,) array — record_name / filename / patient id, same order as `signals`
    dataset_name : "cpsc2018" | "cs" | "csn" | "ptbxl" — determines the S3 key
    strat_fold : (N,) int array, official or MERL train/val/test fold; None otherwise

    Storage layout
    --------------
    Signals are padded to the max length in this dataset and stacked into one
    (N, max_len, 12) float32 array, with a companion (N,) `lengths` array so the
    true (unpadded) length can be recovered on read. Padding uses 0.0 and is
    always trimmed off before the array is handed to a caller — it never reaches
    downstream code as real signal.
    """
    
    N = len(signals)
    if N == 0:
        raise ValueError(f"[{dataset_name}] no signals to write")

    n_leads = signals[0].shape[1]
    for i, sig in enumerate(signals):
        if sig.ndim != 2 or sig.shape[1] != n_leads:
            raise ValueError(
                f"[{dataset_name}] signal {i} has shape {sig.shape}, "
                f"expected (length, {n_leads})"
            )

    lengths = np.array([sig.shape[0] for sig in signals], dtype=np.int32)
    max_len = min(int(lengths.max()), 20000) # Cap at 20000

    padded = np.zeros((N, max_len, n_leads), dtype=np.float32)
    for i, sig in enumerate(signals):
        L = min(sig.shape[0], max_len)
        padded[i, :L, :] = sig[:L, :].astype(np.float32)

    kwargs = {
        "ecg_signals": padded,          # (N, max_len, n_leads) float32, zero-padded
        "lengths":     lengths,         # (N,) int32 — true length before padding
        "labels":      np.asarray(labels), # (N,) int32 — labels 
        "ids":         np.asarray(ids),
    }
    if strat_fold is not None:
        kwargs["strat_fold"] = np.asarray(strat_fold, dtype=np.int32)

    buf = io.BytesIO()
    np.savez_compressed(buf, **kwargs)
    buf.seek(0)
    s3_key = _signals_npz_key(dataset_name)
    with s3_fs.open(f"s3://{bucket_out}/{s3_key}", "wb") as f:
        f.write(buf.read())

    print(
        f"[{dataset_name}] wrote {N:,} signals → s3://{bucket_out}/{s3_key}  "
        f"(max_len={max_len}, lengths range=[{lengths.min()}, {lengths.max()}])"
    )

In [7]:
# Read results from npz files
def read_ecg_signals_from_s3(dataset_name: str) -> dict:
    """
    Returns
    -------
    dict with:
        signals    : list[np.ndarray]  — one (length_i, 12) array per record,
                                          padding trimmed off
        labels     : (N,) int array - label per record
        ids        : (N,) array
        strat_fold : (N,) int array 
    """
    s3_path = f"s3://{bucket_out}/{_signals_npz_key(dataset_name)}"
    with s3_fs.open(s3_path, "rb") as f:
        data = np.load(io.BytesIO(f.read()))

        padded  = data["ecg_signals"]   # (N, max_len, n_leads)
        lengths = data["lengths"]       # (N,)
        signals = [padded[i, :lengths[i], :] for i in range(len(padded))]

        result = {
            "signals": signals,
            "labels":  data["labels"],
            "ids":     data["ids"],
        }
        if "strat_fold" in data:
            result["strat_fold"] = data["strat_fold"]

    print(f"[{dataset_name}] loaded {len(signals):,} signals ← {s3_path}")
    return result

# PTB-XL

In [9]:
ptbxl_csv_filekey = f"{base_prefixes['ptbxl']}/ptbxl_database.csv"

df_ptb_db = read_csv_from_s3(bucket_name, ptbxl_csv_filekey, index_col='ecg_id')
df_ptb_db.columns

Index(['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site',
       'device', 'recording_date', 'report', 'scp_codes', 'heart_axis',
       'infarction_stadium1', 'infarction_stadium2', 'validated_by',
       'second_opinion', 'initial_autogenerated_report', 'validated_by_human',
       'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems',
       'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr'],
      dtype='object')

In [10]:
df_ptb_db = df_ptb_db[['patient_id', 'recording_date', 'scp_codes', 'strat_fold', 'filename_hr']]

In [11]:
# Number of ecg files
df_ptb_db['filename_hr'].nunique(),  len(df_ptb_db)

(21799, 21799)

In [12]:
df_ptb_db.head()

,patient_id,recording_date,scp_codes,strat_fold,filename_hr
ecg_id,,,,,
1,15709.0,1984-11-09 09:17:34,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",3,records500/00000/00001_hr
2,13243.0,1984-11-14 12:55:37,"{'NORM': 80.0, 'SBRAD': 0.0}",2,records500/00000/00002_hr
3,20372.0,1984-11-15 12:49:10,"{'NORM': 100.0, 'SR': 0.0}",5,records500/00000/00003_hr
4,17014.0,1984-11-15 13:44:57,"{'NORM': 100.0, 'SR': 0.0}",3,records500/00000/00004_hr
5,17448.0,1984-11-17 10:43:15,"{'NORM': 100.0, 'SR': 0.0}",4,records500/00000/00005_hr


In [13]:
df_ptb_db['scp_codes'] = df_ptb_db['scp_codes'].apply(lambda x: ast.literal_eval(x))

In [14]:
df_ptb_db.head()

,patient_id,recording_date,scp_codes,strat_fold,filename_hr
ecg_id,,,,,
1,15709.0,1984-11-09 09:17:34,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",3,records500/00000/00001_hr
2,13243.0,1984-11-14 12:55:37,"{'NORM': 80.0, 'SBRAD': 0.0}",2,records500/00000/00002_hr
3,20372.0,1984-11-15 12:49:10,"{'NORM': 100.0, 'SR': 0.0}",5,records500/00000/00003_hr
4,17014.0,1984-11-15 13:44:57,"{'NORM': 100.0, 'SR': 0.0}",3,records500/00000/00004_hr
5,17448.0,1984-11-17 10:43:15,"{'NORM': 100.0, 'SR': 0.0}",4,records500/00000/00005_hr


In [15]:
ptbxl_scpcsv_filekey = f"{base_prefixes['ptbxl']}/scp_statements.csv"

df_ptb_scp = read_csv_from_s3(bucket_name, ptbxl_scpcsv_filekey, index_col=0)
df_ptb_scp.columns

Index(['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class',
       'diagnostic_subclass', 'Statement Category',
       'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code',
       'DICOM Code'],
      dtype='object')

In [16]:
df_ptb_scp.head()

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [17]:
df_ptb_scp['diagnostic_class'].unique()

array(['STTC', 'NORM', 'MI', 'HYP', 'CD', nan], dtype=object)

In [18]:
df_ptb_scp.index.unique()

Index(['NDT', 'NST_', 'DIG', 'LNGQT', 'NORM', 'IMI', 'ASMI', 'LVH', 'LAFB',
       'ISC_', 'IRBBB', '1AVB', 'IVCD', 'ISCAL', 'CRBBB', 'CLBBB', 'ILMI',
       'LAO/LAE', 'AMI', 'ALMI', 'ISCIN', 'INJAS', 'LMI', 'ISCIL', 'LPFB',
       'ISCAS', 'INJAL', 'ISCLA', 'RVH', 'ANEUR', 'RAO/RAE', 'EL', 'WPW',
       'ILBBB', 'IPLMI', 'ISCAN', 'IPMI', 'SEHYP', 'INJIN', 'INJLA', 'PMI',
       '3AVB', 'INJIL', '2AVB', 'ABQRS', 'PVC', 'STD_', 'VCLVH', 'QWAVE',
       'LOWT', 'NT_', 'PAC', 'LPR', 'INVT', 'LVOLT', 'HVOLT', 'TAB_', 'STE_',
       'PRC(S)', 'SR', 'AFIB', 'STACH', 'SARRH', 'SBRAD', 'PACE', 'SVARR',
       'BIGU', 'AFLT', 'SVTAC', 'PSVT', 'TRIGU'],
      dtype='object')

In [19]:
df_ptb_scp['diagnostic'].value_counts()

diagnostic
1.0    44
Name: count, dtype: int64

In [20]:
# Create a dictionary of diagnostic classes for mapping
diagnostic_class_mapping = df_ptb_scp[df_ptb_scp['diagnostic_class'].notna()]['diagnostic_class'].to_dict()
print(diagnostic_class_mapping)

# Function to extract and aggregate diagnostic classes for each patient
def get_diagnostic_classes(scp_codes):
    classes = set()
    for code in scp_codes.keys():
        if code in diagnostic_class_mapping:
            classes.add(diagnostic_class_mapping[code])
    return list(classes)

{'NDT': 'STTC', 'NST_': 'STTC', 'DIG': 'STTC', 'LNGQT': 'STTC', 'NORM': 'NORM', 'IMI': 'MI', 'ASMI': 'MI', 'LVH': 'HYP', 'LAFB': 'CD', 'ISC_': 'STTC', 'IRBBB': 'CD', '1AVB': 'CD', 'IVCD': 'CD', 'ISCAL': 'STTC', 'CRBBB': 'CD', 'CLBBB': 'CD', 'ILMI': 'MI', 'LAO/LAE': 'HYP', 'AMI': 'MI', 'ALMI': 'MI', 'ISCIN': 'STTC', 'INJAS': 'MI', 'LMI': 'MI', 'ISCIL': 'STTC', 'LPFB': 'CD', 'ISCAS': 'STTC', 'INJAL': 'MI', 'ISCLA': 'STTC', 'RVH': 'HYP', 'ANEUR': 'STTC', 'RAO/RAE': 'HYP', 'EL': 'STTC', 'WPW': 'CD', 'ILBBB': 'CD', 'IPLMI': 'MI', 'ISCAN': 'STTC', 'IPMI': 'MI', 'SEHYP': 'HYP', 'INJIN': 'MI', 'INJLA': 'MI', 'PMI': 'MI', '3AVB': 'CD', 'INJIL': 'MI', '2AVB': 'CD'}


In [21]:
df_ptb_db['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

In [22]:
# Map diagnostic classes

df_ptb_db['diagnostic_class_list'] = df_ptb_db['scp_codes'].apply(get_diagnostic_classes)

df_ptb_db['diagnostic_class_list']

ecg_id
1        [NORM]
2        [NORM]
3        [NORM]
4        [NORM]
5        [NORM]
          ...  
21833    [STTC]
21834    [NORM]
21835    [STTC]
21836    [NORM]
21837    [NORM]
Name: diagnostic_class_list, Length: 21799, dtype: object

In [23]:
df_ptb_db['diagnostic_class_list'].value_counts()

diagnostic_class_list
[NORM]                 9069
[MI]                   2532
[STTC]                 2400
[CD]                   1708
[MI, CD]               1297
[STTC, HYP]             781
[MI, STTC]              599
[HYP]                   535
[STTC, CD]              471
[]                      411
[NORM, CD]              407
[MI, STTC, HYP]         361
[HYP, CD]               300
[MI, STTC, CD]          223
[STTC, HYP, CD]         211
[MI, HYP]               183
[MI, STTC, HYP, CD]     156
[MI, HYP, CD]           117
[STTC, NORM]             28
[STTC, NORM, CD]          5
[NORM, HYP, CD]           2
[NORM, HYP]               2
[MI, NORM, HYP, CD]       1
Name: count, dtype: int64

In [24]:
# How many have multiple labels

df_ptb_db[df_ptb_db['diagnostic_class_list'].apply(lambda x: len(x) > 1)]['diagnostic_class_list'].value_counts()

diagnostic_class_list
[MI, CD]               1297
[STTC, HYP]             781
[MI, STTC]              599
[STTC, CD]              471
[NORM, CD]              407
[MI, STTC, HYP]         361
[HYP, CD]               300
[MI, STTC, CD]          223
[STTC, HYP, CD]         211
[MI, HYP]               183
[MI, STTC, HYP, CD]     156
[MI, HYP, CD]           117
[STTC, NORM]             28
[STTC, NORM, CD]          5
[NORM, HYP, CD]           2
[NORM, HYP]               2
[MI, NORM, HYP, CD]       1
Name: count, dtype: int64

In [25]:
# How many have multiple labels

len(df_ptb_db[df_ptb_db['diagnostic_class_list'].apply(lambda x: len(x) > 1)])

5144

In [26]:
# Drop the files with multiple labels or no labels; (HeartLang multilabeled: don't drop)
df_ptb_db = df_ptb_db[df_ptb_db['diagnostic_class_list'].apply(lambda x: len(x) == 1)]
print(len(df_ptb_db))


16244


In [27]:
# PTB has 5 classes: NORM, MI, STTC, CD, HYP
class_map = {
    'NORM': 0, # Normal
    'MI': 1,    # Myocardial Infarction  
    'STTC': 2,  # ST-T Changes
    'CD': 3,    # Conduction Disturbances
    'HYP': 4,   # Hypertrophy
}


In [28]:
df_ptb_db['diagnostic_class_list']

ecg_id
1        [NORM]
2        [NORM]
3        [NORM]
4        [NORM]
5        [NORM]
          ...  
21833    [STTC]
21834    [NORM]
21835    [STTC]
21836    [NORM]
21837    [NORM]
Name: diagnostic_class_list, Length: 16244, dtype: object

In [29]:
# Map classes to label 0..num_classes-1
df_ptb_db['label'] = df_ptb_db['diagnostic_class_list'].apply(lambda x: class_map[x[0]])

In [30]:
df_ptb_db['label'].value_counts()

label
0    9069
1    2532
2    2400
3    1708
4     535
Name: count, dtype: int64

In [31]:
df_ptb_db['diagnostic_class_list'].value_counts()

diagnostic_class_list
[NORM]    9069
[MI]      2532
[STTC]    2400
[CD]      1708
[HYP]      535
Name: count, dtype: int64

In [32]:
# Read ecg data at 500 Hz from filename_hr files

# Function to read ECG signal data from s3
def read_ecg_data_ptb(bucket: str, key: str) -> tuple[np.ndarray, dict] | None:
    dat_key = f"{base_prefixes['ptbxl']}/{key}.dat"
    hea_key = f"{base_prefixes['ptbxl']}/{key}.hea"
    tmp_base = key.split('/')[-1]
    with tempfile.TemporaryDirectory() as tmp:
        for keyname, filename in [(dat_key, f"{tmp_base}.dat"), (hea_key, f"{tmp_base}.hea")]:
            s3_client.download_file(bucket, keyname, os.path.join(tmp, filename))
        try:
            rec = wfdb.rdrecord(os.path.join(tmp, tmp_base))
            header = wfdb.rdheader(os.path.join(tmp, tmp_base)).__dict__
            return rec.p_signal, header

        except Exception as e:
            print(f"wfdb failed on {key}: {e}")
            return None

In [33]:
# Test on one file
read_ecg_data_ptb(bucket_name, df_ptb_db['filename_hr'].values[0])

(array([[-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        ...,
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ]],
       shape=(5000, 12)),
 {'record_name': '00001_hr',
  'n_sig': 12,
  'fs': 500,
  'counter_freq': None,
  'base_counter': None,
  'sig_len': 5000,
  'base_time': None,
  'base_date': None,
  'comments': [],
  'sig_name': ['I',
   'II',
   'III',
   'AVR',
   'AVL',
   'AVF',
   'V1',
   'V2',
   'V3',
   'V4',
   'V5',
   'V6'],
  'p_signal': None,
  'd_signal': None,
  'e_p_signal': None,
  'e_d_signal': None,
  'file_name': ['00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00001_hr.dat',
   '00

In [34]:
read_ecg_data_ptb(bucket_name, df_ptb_db['filename_hr'].values[0])[0].shape

(5000, 12)

In [35]:
# Test on two files
df_ptb_db.loc[:2, 'filename_hr'].apply(lambda x: read_ecg_data_ptb(bucket_name, x)[0])


ecg_id
1    [[-0.115, -0.05, 0.065, 0.082, -0.09, 0.007, -...
2    [[-0.015, 0.12, 0.135, -0.053, -0.075, 0.127, ...
Name: filename_hr, dtype: object

In [36]:
# Test on two files
df_ptb_db.loc[:2, 'filename_hr'].apply(lambda x: read_ecg_data_ptb(bucket_name, x)[1])


ecg_id
1    {'record_name': '00001_hr', 'n_sig': 12, 'fs':...
2    {'record_name': '00002_hr', 'n_sig': 12, 'fs':...
Name: filename_hr, dtype: object

In [37]:
# Read all ecgs

all_ecgs = []
for filename_hr in tqdm(df_ptb_db['filename_hr'].values):
    read_ecg_result = read_ecg_data_ptb(bucket_name, filename_hr)
    ecg_12lead = np.array(read_ecg_result[0]) if read_ecg_result is not None else None
    all_ecgs.append(ecg_12lead)


100%|██████████| 16244/16244 [51:33<00:00,  5.25it/s] 


In [38]:
# all signals have length 5000
# Signal lengths
lens = []
for ecg in all_ecgs:
    lens.append(len(ecg))

np.unique(lens, return_counts=True)

(array([5000]), array([16244]))

In [39]:
# Save to s3

write_ecg_signals_to_s3(
    signals=all_ecgs,
    labels=df_ptb_db['label'].to_numpy(),
    ids=df_ptb_db['patient_id'].to_numpy(),
    dataset_name="ptbxl",
    strat_fold=df_ptb_db["strat_fold"].to_numpy()
)

# remove from memory
del all_ecgs
del df_ptb_db

[ptbxl] wrote 16,244 signals → s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/ptbxl_signals.npz  (max_len=5000, lengths range=[5000, 5000])


In [40]:
# Check saved file
data = read_ecg_signals_from_s3(dataset_name="ptbxl") 

[ptbxl] loaded 16,244 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/ptbxl_signals.npz


In [41]:
# Read one ecg
data['signals'][0], data['ids'][0]

(array([[-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        ...,
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ]],
       shape=(5000, 12), dtype=float32),
 np.float64(15709.0))

In [42]:
# Free memory
del data

# CPSC2018

In [8]:
# Function to list all files in nested folders automatically
def _list_all_files(bucket: str, prefix: str) -> list:
    files = []
    paginator = s3_client.get_paginator('list_objects_v2') #gives an iterator that automatically requests every page
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        files.extend([content['Key'] for content in page.get('Contents', [])]) #add full path of each object to files
    mat_file_names = [file for file in files if file.endswith('.mat')]
    random.seed(42)
    random.shuffle(mat_file_names)
    return mat_file_names


# Function to read ECG signal data from s3
def _read_data(bucket: str, mat_key: str) -> tuple[np.ndarray, list[str], dict] | None:
    hea_key = mat_key.replace(".mat", ".hea")
    base = os.path.basename(mat_key).split(".")[0]
    with tempfile.TemporaryDirectory() as tmp:
        for key, name in [(mat_key, f"{base}.mat"), (hea_key, f"{base}.hea")]:
            s3_client.download_file(bucket, key, os.path.join(tmp, name))
        try:
            recordname = os.path.join(tmp, base) 
            rec = wfdb.rdrecord(recordname)
            header = wfdb.rdheader(recordname)

            # Return metadata dict
            metadata = {
                "header": header.__dict__,  # structured wfdb header fields
            }

            return rec.p_signal, metadata

        except Exception as e:
            print(f"wfdb failed on {mat_key}: {e}")
            return None

In [44]:
# All ecg files
cpsc_files = _list_all_files(bucket_name, base_prefixes['cpsc_2018'])

In [45]:
len(cpsc_files)

6877

In [46]:
# Extract all signals and metadata from all ecg files
all_signals, all_metadata = [], []
for mat_key in tqdm(cpsc_files):
    signal, metadata = _read_data(bucket_name, mat_key)
    all_signals.append(signal)
    all_metadata.append(metadata)

100%|██████████| 6877/6877 [21:35<00:00,  5.31it/s]


In [47]:
# Print one signal and metadata
all_signals[0], all_metadata[0]

(array([[-7.000e-03, -1.190e-01, -1.120e-01, ..., -1.440e-01, -4.900e-02,
         -3.821e+00],
        [-3.000e-03, -1.750e-01, -1.720e-01, ..., -2.060e-01, -7.300e-02,
         -5.388e+00],
        [-7.000e-03, -1.570e-01, -1.500e-01, ..., -1.870e-01, -6.300e-02,
         -4.804e+00],
        ...,
        [-2.900e-02, -8.300e-02, -5.400e-02, ..., -8.400e-02, -9.800e-02,
         -2.800e-02],
        [-3.600e-02, -8.300e-02, -4.700e-02, ..., -7.500e-02, -8.800e-02,
         -2.600e-02],
        [-4.200e-02, -8.700e-02, -4.500e-02, ..., -6.600e-02, -8.700e-02,
         -2.600e-02]], shape=(7476, 12)),
 {'header': {'record_name': 'A6163',
   'n_sig': 12,
   'fs': 500,
   'counter_freq': None,
   'base_counter': None,
   'sig_len': 7476,
   'base_time': datetime.time(0, 0, 12),
   'base_date': None,
   'comments': ['Age: 34',
    'Sex: Female',
    'Dx: 429622005',
    'Rx: Unknown',
    'Hx: Unknown',
    'Sx: Unknown'],
   'sig_name': ['I',
    'II',
    'III',
    'aVR',
    'aVL',
  

In [48]:
# Print one metadata header format
all_metadata[0]['header']

{'record_name': 'A6163',
 'n_sig': 12,
 'fs': 500,
 'counter_freq': None,
 'base_counter': None,
 'sig_len': 7476,
 'base_time': datetime.time(0, 0, 12),
 'base_date': None,
 'comments': ['Age: 34',
  'Sex: Female',
  'Dx: 429622005',
  'Rx: Unknown',
  'Hx: Unknown',
  'Sx: Unknown'],
 'sig_name': ['I',
  'II',
  'III',
  'aVR',
  'aVL',
  'aVF',
  'V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6'],
 'p_signal': None,
 'd_signal': None,
 'e_p_signal': None,
 'e_d_signal': None,
 'file_name': ['A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat',
  'A6163.mat'],
 'fmt': ['16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16'],
 'samps_per_frame': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'skew': [None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None],
 'byte_offset': [24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,

In [49]:
# Extract diagnostic code and record name
all_metadata[0]['header']['comments'], all_metadata[0]['header']['record_name']

(['Age: 34',
  'Sex: Female',
  'Dx: 429622005',
  'Rx: Unknown',
  'Hx: Unknown',
  'Sx: Unknown'],
 'A6163')

In [50]:
# Extract diagnostic code
for substring in all_metadata[0]['header']['comments']:
    if substring.startswith('Dx'):
        codes = re.findall(r"\d+", substring)
        diag_codes = [int(code.strip()) for code in codes]
        print(diag_codes)

[429622005]


In [51]:
# Labels
SNOMED_TO_LABEL = {
    '426783006': 'Normal',
    '164889003': 'AF',
    '270492004': 'I-AVB',
    '164909002': 'LBBB',
    '59118001': 'RBBB',
    '284470004': 'PAC',
    '164884008': 'PVC',
    '429622005': 'STD',
    '164931005': 'STE'
}

In [52]:
# Extract diagnostic codes and record names, age and sex from all metadata

record_names = []
diag_codes = []
labels = []
ages = []
sexes = []
for metadata in all_metadata:
    record_names.append(metadata['header']['record_name'])
    # Extract diagnostic codes
    for substring in metadata['header']['comments']:
        if substring.startswith('Dx'):
            codes = re.findall(r"\d+", substring)
            codes_list = [code.strip() for code in codes]
            diag_codes.append(codes_list)
            # Map to labels
            labels_codes = [SNOMED_TO_LABEL.get(code, "Unknown") for code in codes_list]
            labels.append(labels_codes)
        if substring.lower().startswith('age'):
            age = substring.split(':')[1].strip()
            ages.append(age)
        if substring.lower().startswith('sex'):
            sex = substring.split(':')[1].strip().lower()
            sexes.append(sex)

In [53]:
record_names[:2], diag_codes[:2], labels[:2], ages[:2], sexes[:2]

(['A6163', 'A2735'],
 [['429622005'], ['284470004']],
 [['STD'], ['PAC']],
 ['34', '22'],
 ['female', 'male'])

In [54]:
# Check that lengths of arrays align
len(record_names), len(diag_codes), len(labels), len(ages), len(sexes), len(all_signals)

(6877, 6877, 6877, 6877, 6877, 6877)

In [55]:
# Check ecg signal shapes
siglens = []
nleads = []
for sig in all_signals:
    siglens.append(len(sig))
    nleads.append(np.array(sig).shape[1])

In [56]:
# All have 12 leads
np.unique(nleads)

array([12])

In [57]:
# Range of signal lengths
min(siglens), max(siglens), np.unique(siglens, return_counts=True)

(3000,
 72000,
 (array([ 3000,  4000,  4500, ..., 66000, 69000, 72000], shape=(1650,)),
  array([1, 1, 4, ..., 1, 1, 1], shape=(1650,))))

In [58]:
# How many signal lengths are standard lengths, out of total
np.count_nonzero(np.array(siglens)==5000), len(siglens)

(2416, 6877)

In [59]:
# How many signal lengths are longer than 5000
np.count_nonzero(np.array(siglens)>5000)

4451

In [60]:
# How many signal lengths are longer than 10000
np.count_nonzero(np.array(siglens)>10000)

1206

In [61]:
# How many signal lengths are longer than 20000
np.count_nonzero(np.array(siglens)>20000)

243

In [62]:
df_cpsc = pd.DataFrame({'record_name': record_names,
                       'age': ages,
                       'sex': sexes,
                       'ecg_signals': all_signals, 
                       'diag_code': labels,
                       'snomed_codes': diag_codes})

In [63]:
df_cpsc.head()

,record_name,age,sex,ecg_signals,diag_code,snomed_codes
0,A6163,34,female,"[[-0.007, -0.119, -0.112, 0.063, 0.052, -0.116...",[STD],[429622005]
1,A2735,22,male,"[[0.042, 0.001, -0.041, -0.021, 0.041, -0.02, ...",[PAC],[284470004]
2,A5048,52,male,"[[0.24, 0.986, 0.746, -0.612, -0.253, 0.866, -...",[STD],[429622005]
3,A2298,58,female,"[[0.398, 0.359, -0.04, -0.378, 0.219, 0.159, -...",[STD],[429622005]
4,A1486,29,female,"[[-0.037, -0.038, -0.001, 0.038, -0.019, -0.01...",[STD],[429622005]


In [64]:
# Length of dataframe
len(df_cpsc)

6877

In [65]:
# Drop all rows with multiple codes

df_cpsc = df_cpsc[df_cpsc['diag_code'].apply(lambda x: len(x) == 1)]
len(df_cpsc)

6401

In [66]:
df_cpsc['diag_code'] = df_cpsc['diag_code'].apply(lambda x: x[0])

In [67]:
df_cpsc['diag_code'].value_counts()

diag_code
RBBB      1533
AF         976
Normal     918
STD        784
I-AVB      686
PVC        607
PAC        533
STE        185
LBBB       179
Name: count, dtype: int64

In [68]:
# Numerical label mapping
# Labels
CODE_TO_LABEL = {code: label for label, code in enumerate(df_cpsc['diag_code'].unique())}
print(CODE_TO_LABEL)

{'STD': 0, 'PAC': 1, 'AF': 2, 'PVC': 3, 'LBBB': 4, 'RBBB': 5, 'Normal': 6, 'I-AVB': 7, 'STE': 8}


In [69]:
df_cpsc['label'] = df_cpsc['diag_code'].map(CODE_TO_LABEL)

In [70]:
df_cpsc.head()

,record_name,age,sex,ecg_signals,diag_code,snomed_codes,label
0,A6163,34,female,"[[-0.007, -0.119, -0.112, 0.063, 0.052, -0.116...",STD,[429622005],0
1,A2735,22,male,"[[0.042, 0.001, -0.041, -0.021, 0.041, -0.02, ...",PAC,[284470004],1
2,A5048,52,male,"[[0.24, 0.986, 0.746, -0.612, -0.253, 0.866, -...",STD,[429622005],0
3,A2298,58,female,"[[0.398, 0.359, -0.04, -0.378, 0.219, 0.159, -...",STD,[429622005],0
4,A1486,29,female,"[[-0.037, -0.038, -0.001, 0.038, -0.019, -0.01...",STD,[429622005],0


In [71]:
df_cpsc['label'].value_counts()

label
5    1533
2     976
6     918
0     784
7     686
3     607
1     533
8     185
4     179
Name: count, dtype: int64

In [72]:
# Save to s3

write_ecg_signals_to_s3(
    signals=df_cpsc['ecg_signals'].to_list(),
    labels=df_cpsc['label'].to_list(),
    ids=df_cpsc['record_name'].to_list(),
    dataset_name="cpsc2018",
    strat_fold=None
)

# remove from memory
del all_signals
del df_cpsc

[cpsc2018] wrote 6,401 signals → s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/cpsc2018_signals.npz  (max_len=20000, lengths range=[3000, 72000])


In [78]:
# Check saved file
data = read_ecg_signals_from_s3(dataset_name="cpsc2018") 

[cpsc2018] loaded 6,401 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/cpsc2018_signals.npz


In [79]:
np.unique(data['labels'], return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8]),
 array([ 784,  533,  976,  607,  179, 1533,  918,  686,  185]))

In [80]:
# Check an ecg
data['signals'][0], data['ids'][0], data['labels'][0]

(array([[-7.000e-03, -1.190e-01, -1.120e-01, ..., -1.440e-01, -4.900e-02,
         -3.821e+00],
        [-3.000e-03, -1.750e-01, -1.720e-01, ..., -2.060e-01, -7.300e-02,
         -5.388e+00],
        [-7.000e-03, -1.570e-01, -1.500e-01, ..., -1.870e-01, -6.300e-02,
         -4.804e+00],
        ...,
        [-2.900e-02, -8.300e-02, -5.400e-02, ..., -8.400e-02, -9.800e-02,
         -2.800e-02],
        [-3.600e-02, -8.300e-02, -4.700e-02, ..., -7.500e-02, -8.800e-02,
         -2.600e-02],
        [-4.200e-02, -8.700e-02, -4.500e-02, ..., -6.600e-02, -8.700e-02,
         -2.600e-02]], shape=(7476, 12), dtype=float32),
 np.str_('A6163'),
 np.int64(0))

In [81]:
# Free memory
del data

# Chapman Shaoxing Ningbo

In [9]:
# List all files for csn
csn_files = _list_all_files(bucket_name, base_prefixes["csn"])

In [10]:
print(len(csn_files), csn_files[0:3])

45152 ['data/csn/WFDBRecords/16/163/JS15732.mat', 'data/csn/WFDBRecords/03/037/JS02868.mat', 'data/csn/WFDBRecords/10/107/JS10144.mat']


In [11]:
csn_diag_key = f"{base_prefixes['csn']}/ConditionNames_SNOMED-CT.csv"
df_csn_codes = read_csv_from_s3(bucket_name, csn_diag_key, index_col=None)

In [12]:
df_csn_codes.head()

,Acronym Name,Full Name,Snomed_CT
0,1AVB,1 degree atrioventricular block,270492004
1,2AVB,2 degree atrioventricular block,195042002
2,2AVB1,2 degree atrioventricular block(Type one),54016002
3,2AVB2,2 degree atrioventricular block(Type two),28189009
4,3AVB,3 degree atrioventricular block,27885002


In [13]:
df_csn_codes['Snomed_CT'].unique(), df_csn_codes['Snomed_CT'].nunique()

(array([270492004, 195042002,  54016002,  28189009,  27885002, 251173003,
         39732003, 284470004, 164917005,  47665007, 233917008, 251199005,
        251198002, 428417006, 164942001, 698252002, 426995002, 251164006,
        164909002, 164873001, 251146004, 251148003, 251147008, 164865005,
        164947007, 164912004, 111975006, 446358003,  59118001,  89792004,
        429622005, 164930006, 428750005, 164931005, 164934002,  59931005,
        164937009,  11157007,  75532003,  13640000,  17338001, 195060002,
        251180001, 195101003,  74390002, 426177001, 426783006, 164889003,
        427084000, 164890007, 427393009, 426761007, 713422000, 233896004,
        233897008]),
 55)

In [14]:
df_csn_codes['Acronym Name'].unique(), df_csn_codes['Acronym Name'].nunique() 

(array(['1AVB', '2AVB', '2AVB1', '2AVB2', '3AVB', 'ABI', 'ALS', 'APB',
        'AQW', 'ARS', 'AVB', 'CCR', 'CR', 'ERV', 'FQRS', 'IDC', 'IVB',
        'JEB', 'JPT', 'LBBB', 'LBBBB', 'LFBBB', 'LVH', 'LVQRSAL',
        'LVQRSCL', 'LVQRSLL', 'MI', 'MIBW', 'MIFW', 'MILW', 'MISW', 'PRIE',
        'PWC', 'QTIE', 'RAH', 'RBBB', 'RVH', 'STDD', 'STE', 'STTC', 'STTU',
        'TWC', 'TWO', 'UW', 'VB', 'VEB', 'VFW', 'VPB', 'VPE', 'VET',
        'WAVN', 'WPW', 'SB', 'SR', 'AFIB', 'ST', 'AF', 'SA', 'SVT', 'AT',
        'AVNRT', 'AVRT', 'SAAWR'], dtype=object),
 63)

In [15]:
# Map SNOMED to Acronym Name code, each SNOMED can have multiple Acronym Name codes
from collections import defaultdict
diag_code_map = defaultdict(list)

for snomed, acronym in zip(df_csn_codes['Snomed_CT'], df_csn_codes['Acronym Name']):
    diag_code_map[snomed].append(acronym)

In [16]:
diag_code_map

defaultdict(list,
            {270492004: ['1AVB'],
             195042002: ['2AVB'],
             54016002: ['2AVB1'],
             28189009: ['2AVB2'],
             27885002: ['3AVB'],
             251173003: ['ABI'],
             39732003: ['ALS'],
             284470004: ['APB'],
             164917005: ['AQW'],
             47665007: ['ARS'],
             233917008: ['AVB'],
             251199005: ['CCR'],
             251198002: ['CR'],
             428417006: ['ERV'],
             164942001: ['FQRS'],
             698252002: ['IDC', 'IVB'],
             426995002: ['JEB'],
             251164006: ['JPT'],
             164909002: ['LBBB', 'LBBBB', 'LFBBB'],
             164873001: ['LVH'],
             251146004: ['LVQRSAL'],
             251148003: ['LVQRSCL'],
             251147008: ['LVQRSLL'],
             164865005: ['MI', 'MIBW', 'MIFW', 'MILW', 'MISW'],
             164947007: ['PRIE'],
             164912004: ['PWC'],
             111975006: ['QTIE'],
             44635

In [ ]:
# Extract all signals and metadata from all ecg files
all_signals, all_metadata = [], []
exs = []
for mat_key in tqdm(csn_files):
    try:
        signal, metadata = _read_data(bucket_name, mat_key)
        all_signals.append(signal)
        all_metadata.append(metadata)
    except Exception as e:
        exs.append(e)

 35%|███▌      | 15940/45152 [52:09<1:41:41,  4.79it/s]

wfdb failed on data/csn/WFDBRecords/23/236/JS23074.mat: list index out of range


 86%|████████▌ | 38622/45152 [2:06:45<23:03,  4.72it/s]  

wfdb failed on data/csn/WFDBRecords/01/019/JS01052.mat: time data '/' does not match format '%d/%m/%Y'


100%|██████████| 45152/45152 [2:28:49<00:00,  5.06it/s]


In [ ]:
print(exs, len(exs), len(all_signals))

[TypeError('cannot unpack non-iterable NoneType object'), TypeError('cannot unpack non-iterable NoneType object')] 2 45150


In [ ]:
# Print one signal and metadata
all_signals[0], all_metadata[0]

(array([[ 0.098,  0.02 , -0.078, ..., -0.024, -0.044,  0.005],
        [ 0.098,  0.02 , -0.078, ..., -0.024, -0.044,  0.   ],
        [ 0.098,  0.015, -0.083, ..., -0.024, -0.049,  0.   ],
        ...,
        [-0.41 , -0.229,  0.181, ...,  0.215, -0.293, -0.249],
        [-0.42 , -0.234,  0.185, ...,  0.215, -0.293, -0.249],
        [-0.41 , -0.254,  0.156, ...,  0.21 , -0.303, -0.259]],
       shape=(5000, 12)),
 {'header': {'record_name': 'JS15732',
   'n_sig': 12,
   'fs': 500,
   'counter_freq': None,
   'base_counter': None,
   'sig_len': 5000,
   'base_time': None,
   'base_date': None,
   'comments': ['Age: 89',
    'Sex: Male',
    'Dx: 164890007,429622005,55930002,59931005',
    'Rx: Unknown',
    'Hx: Unknown',
    'Sx: Unknown'],
   'sig_name': ['I',
    'II',
    'III',
    'aVR',
    'aVL',
    'aVF',
    'V1',
    'V2',
    'V3',
    'V4',
    'V5',
    'V6'],
   'p_signal': None,
   'd_signal': None,
   'e_p_signal': None,
   'e_d_signal': None,
   'file_name': ['JS1573

In [ ]:
# Print one metadata header format
all_metadata[0]['header']

{'record_name': 'JS15732',
 'n_sig': 12,
 'fs': 500,
 'counter_freq': None,
 'base_counter': None,
 'sig_len': 5000,
 'base_time': None,
 'base_date': None,
 'comments': ['Age: 89',
  'Sex: Male',
  'Dx: 164890007,429622005,55930002,59931005',
  'Rx: Unknown',
  'Hx: Unknown',
  'Sx: Unknown'],
 'sig_name': ['I',
  'II',
  'III',
  'aVR',
  'aVL',
  'aVF',
  'V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6'],
 'p_signal': None,
 'd_signal': None,
 'e_p_signal': None,
 'e_d_signal': None,
 'file_name': ['JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat',
  'JS15732.mat'],
 'fmt': ['16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16',
  '16'],
 'samps_per_frame': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'skew': [None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None],
 'byte_offset': [24, 24, 24

In [ ]:
# Extract diagnostic code and record name
all_metadata[0]['header']['comments'], all_metadata[0]['header']['record_name']

(['Age: 89',
  'Sex: Male',
  'Dx: 164890007,429622005,55930002,59931005',
  'Rx: Unknown',
  'Hx: Unknown',
  'Sx: Unknown'],
 'JS15732')

In [ ]:
# Extract diagnostic code
for substring in all_metadata[0]['header']['comments']:
    if substring.startswith('Dx'):
        codes = re.findall(r"\d+", substring)
        diag_codes = [int(code.strip()) for code in codes]
        print(diag_codes)

[164890007, 429622005, 55930002, 59931005]


In [ ]:
# Extract diagnostic codes and record names, age and sex from all metadata

record_names = []
diag_codes = []
codenames = []
ages = []
sexes = []
for metadata in all_metadata:
    record_names.append(metadata['header']['record_name'])
    # Extract diagnostic codes
    for substring in metadata['header']['comments']:
        if substring.startswith('Dx'):
            codes = re.findall(r"\d+", substring)
            codes_list = [int(code.strip()) for code in codes]
            diag_codes.append(codes_list)
            # Map to codes
            # Flatten acronyms directly
            named_codes = []
            for code in codes_list:
                acronyms = diag_code_map.get(code, ["Unknown"])
                named_codes.extend(acronyms)
            codenames.append(named_codes)
        if substring.lower().startswith('age'):
            age = substring.split(':')[1].strip()
            ages.append(age)
        if substring.lower().startswith('sex'):
            sex = substring.split(':')[1].strip().lower()
            sexes.append(sex)

In [ ]:
record_names[:2], diag_codes[:2], codenames[:2], ages[:2], sexes[:2]

(['JS15732', 'JS02868'],
 [[164890007, 429622005, 55930002, 59931005], [426177001]],
 [['AF', 'STDD', 'Unknown', 'TWO'], ['SB']],
 ['89', '37'],
 ['male', 'male'])

In [ ]:
assert len(all_signals) == len(codenames) == len(record_names) == len(ages) == len(sexes) == len(diag_codes), (
    len(all_signals), len(codenames), len(record_names), len(ages), len(sexes), len(diag_codes)
)

In [ ]:
# Check ecg signal shapes
siglens = []
nleads = []
for sig in all_signals:
    siglens.append(len(sig))
    nleads.append(np.array(sig).shape[1])

In [ ]:
# All have 12 leads
np.unique(nleads)

array([12])

In [ ]:
# Range of signal lengths
min(siglens), max(siglens), np.unique(siglens, return_counts=True)

(5000, 5000, (array([5000]), array([45150])))

In [ ]:
codenames

[['AF', 'STDD', 'Unknown', 'TWO'],
 ['SB'],
 ['Unknown'],
 ['SB', 'TWC', 'STE'],
 ['APB', 'Unknown', 'TWC', 'SA', 'Unknown'],
 ['SR'],
 ['SR'],
 ['SR'],
 ['ST', 'STDD', 'AQW', 'LVH', 'Unknown'],
 ['1AVB', 'ST', 'Unknown'],
 ['ST'],
 ['SB'],
 ['SR'],
 ['APB', 'Unknown', 'ST', 'AQW', 'TWC'],
 ['SR'],
 ['SB'],
 ['ST'],
 ['ST'],
 ['SB'],
 ['ST', 'Unknown', 'Unknown'],
 ['SB', '1AVB'],
 ['AF', 'TWC', 'Unknown', 'TWO'],
 ['AF', 'Unknown', 'Unknown', 'Unknown'],
 ['SB', 'Unknown'],
 ['STE', 'ST', 'ERV'],
 ['ST'],
 ['AFIB'],
 ['PRIE', 'ST', 'Unknown', 'Unknown'],
 ['ST'],
 ['MI',
  'MIBW',
  'MIFW',
  'MILW',
  'MISW',
  'STDD',
  'TWO',
  'AQW',
  'SR',
  'Unknown',
  'TWC'],
 ['AF'],
 ['ST', 'TWC'],
 ['SB'],
 ['ST'],
 ['AQW', 'ST', 'Unknown', 'Unknown'],
 ['PRIE', 'Unknown', 'STDD', 'TWO', 'SB', 'Unknown'],
 ['SB', '1AVB'],
 ['SR'],
 ['ST'],
 ['ST', 'ALS'],
 ['ST', 'PRIE', 'STTC', 'TWO'],
 ['SB', 'TWC'],
 ['AFIB', 'ALS', 'IDC', 'IVB'],
 ['AF', 'Unknown'],
 ['SB', 'QTIE', 'TWC'],
 ['SB'],
 ['

In [ ]:
# How many codenames have length at least 1 and do not contain Unknown
len([cname for cname in codenames if len(cname)>=1 and 'Unknown' not in cname])

31960

In [ ]:
# How many codenames have multiple labels and do not contain Unknown
len([cname for cname in codenames if len(cname)>1 and 'Unknown' not in cname])

10350

In [ ]:
# How many codenames have 2 labels, one of which is Unknown
len([cname for cname in codenames if len(cname)==2 and 'Unknown' in cname])

4972

In [ ]:
# How many codenames which don't have Unknown
len([cname for cname in codenames if 'Unknown' not in cname])

31960

In [ ]:
# How many codenames have length exactly 1 and do not contain Unknown
len([cname for cname in codenames if len(cname)==1 and 'Unknown' not in cname])

21610

In [ ]:
# Unique code names
flat_codenames = [
    cname
    for sublist in codenames
    for cname in (sublist if isinstance(sublist, list) else [sublist])
    if cname != "Unknown"
]
set_cnames = set(flat_codenames)
print(set_cnames, len(set_cnames))

{'PWC', 'VPB', 'CR', 'AFIB', 'STDD', 'STE', 'VB', 'AQW', 'VFW', 'VET', 'RBBB', 'TWO', 'SA', 'LBBBB', 'STTU', 'ARS', 'VPE', 'VEB', 'MIFW', '1AVB', 'AVB', 'LVQRSAL', 'ABI', 'MI', 'UW', 'WAVN', 'ALS', 'PRIE', 'FQRS', 'ST', 'WPW', 'AVRT', 'AF', 'LVH', 'IVB', 'LFBBB', 'SR', 'STTC', 'CCR', 'LBBB', 'APB', 'MISW', 'IDC', 'ERV', 'AT', 'QTIE', 'JPT', '2AVB', 'RAH', '3AVB', '2AVB1', 'SVT', 'TWC', 'JEB', 'SAAWR', 'MILW', 'MIBW', 'SB', 'RVH'} 59


In [ ]:
# How many times code names appear
from collections import Counter
Counter(flat_codenames)

Counter({'SB': 16559,
         'SR': 8125,
         'AF': 8060,
         'ST': 7254,
         'TWC': 7043,
         'TWO': 2877,
         'SA': 2550,
         'AFIB': 1780,
         'STDD': 1668,
         'ALS': 1545,
         'APB': 1312,
         'STTC': 1158,
         '1AVB': 1140,
         'AQW': 1063,
         'LVQRSAL': 1043,
         'ARS': 853,
         'STE': 801,
         'IDC': 771,
         'IVB': 771,
         'SVT': 724,
         'RBBB': 649,
         'LVH': 647,
         'QTIE': 394,
         'ERV': 366,
         'AT': 297,
         'VPB': 294,
         'AVB': 244,
         'LBBB': 240,
         'LBBBB': 240,
         'LFBBB': 240,
         'STTU': 176,
         'CCR': 162,
         'PWC': 142,
         'UW': 136,
         'MI': 123,
         'MIBW': 123,
         'MIFW': 123,
         'MILW': 123,
         'MISW': 123,
         'VFW': 116,
         'RVH': 110,
         'CR': 76,
         '3AVB': 76,
         'JEB': 75,
         'WPW': 72,
         '2AVB': 66,
         '

In [ ]:
# Drop multiple or unknown labels
# Earlier iteration:
# This drops "Unknown" from labels first, then drops multiple or empty labels,
# but MERL dropped all records containing "Unknown"

# Current iteration:
# Drop all records containing "Unknown"
# Then drop records with multiple labels
# Consistent with MERL but keeps singly-labeled records only

entries = list(zip(all_signals,
                   codenames,
                   record_names,
                   ages,
                   sexes))

# Filter based on codes: remove records with 'Unknown' and select single-labeled records
filtered_entries = [
    (sig, code[0], rec, age, sex)
    for sig, code, rec, age, sex in entries
    if "Unknown" not in code and len(code)==1
]

# Unpack filtered values
filtered_signals, filtered_codenames, filtered_recordnames, filtered_ages, filtered_sexes =  zip(*filtered_entries)

In [ ]:
# Map to numerical labels
unique_codenames = np.unique(np.array(filtered_codenames))

label_map = {codename:i for i, codename in enumerate(unique_codenames)}
labels = np.array([label_map[c] for c in filtered_codenames])

In [ ]:
np.unique(labels, return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]),
 array([   1, 1483,  422,    1,   31,    5, 1234, 8909, 5908, 3223,  390,
           3]))

In [ ]:
np.unique(filtered_codenames, return_counts=True)

(array(['1AVB', 'AF', 'AFIB', 'APB', 'AT', 'AVRT', 'SA', 'SB', 'SR', 'ST',
        'SVT', 'VEB'], dtype='<U4'),
 array([   1, 1483,  422,    1,   31,    5, 1234, 8909, 5908, 3223,  390,
           3]))

In [ ]:
len(labels), len(filtered_signals), len(all_signals)

(21610, 21610, 45150)

In [ ]:
# Save to s3

write_ecg_signals_to_s3(
    signals=list(filtered_signals),
    labels=list(labels),
    ids=list(filtered_recordnames),
    dataset_name="csn",
    strat_fold=None
)

# remove from memory
del all_signals, filtered_signals

[csn] wrote 21,610 signals → s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/csn_signals.npz  (max_len=5000, lengths range=[5000, 5000])


In [43]:
# Check saved file
data = read_ecg_signals_from_s3(dataset_name="csn")

[csn] loaded 21,610 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/csn_signals.npz


In [44]:
np.unique(data['labels'], return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]),
 array([   1, 1483,  422,    1,   31,    5, 1234, 8909, 5908, 3223,  390,
           3]))

In [45]:
# Check an ecg
data['signals'][0], data['ids'][0], data['labels'][0]

(array([[-0.039, -0.029,  0.01 , ..., -0.098, -0.068, -0.059],
        [-0.039, -0.029,  0.01 , ..., -0.098, -0.068, -0.059],
        [-0.039, -0.029,  0.01 , ..., -0.098, -0.068, -0.059],
        ...,
        [ 0.049, -0.054, -0.102, ..., -0.244, -0.176, -0.01 ],
        [ 0.039, -0.054, -0.093, ..., -0.244, -0.176, -0.01 ],
        [ 0.039, -0.063, -0.102, ..., -0.244, -0.176, -0.01 ]],
       shape=(5000, 12), dtype=float32),
 np.str_('JS02868'),
 np.int64(7))

In [46]:
# Free memory
del data

# Chapman Shaoxing

In [47]:
cs_key = "Chapman_Shaoxing/ECGDataDenoised/"
cs_files = []

paginator = s3_client.get_paginator('list_objects_v2')

# Get all csv files
page_iterator = paginator.paginate(
    Bucket=bucket_name,
    Prefix=cs_key
)

for page in page_iterator:
    if 'Contents' in page:
        for obj in page['Contents']:
            cs_files.append(obj['Key'])

In [48]:
# # CS files
print(len(cs_files), cs_files[0:5])

10646 ['Chapman_Shaoxing/ECGDataDenoised/MUSE_20180111_155115_19000.csv', 'Chapman_Shaoxing/ECGDataDenoised/MUSE_20180111_155154_74000.csv', 'Chapman_Shaoxing/ECGDataDenoised/MUSE_20180111_155203_15000.csv', 'Chapman_Shaoxing/ECGDataDenoised/MUSE_20180111_155249_70000.csv', 'Chapman_Shaoxing/ECGDataDenoised/MUSE_20180111_155542_84000.csv']


In [49]:
# Read 1 ECG record
fs = s3fs.S3FileSystem()

with fs.open(f"{bucket_name}/{cs_files[0]}", 'rb') as f:
    df_1 = pd.read_csv(f, header=None, index_col=None)

In [50]:
df_1.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,-165.33,-358.97,-121.710,270.25,-28.869,-231.16,448.39,636.52,618.45,-7.8987,-315.16,-570.84
1,-150.75,-336.81,-114.980,251.41,-23.883,-217.52,441.74,646.47,642.56,35.3890,-269.51,-532.21
2,-136.69,-315.56,-108.630,233.45,-19.126,-204.36,436.06,656.31,665.95,76.5720,-225.83,-495.39
3,-123.74,-296.23,-103.090,217.23,-14.815,-192.29,432.27,666.14,688.05,113.5100,-186.00,-461.84
4,-112.57,-279.75,-98.611,203.53,-11.213,-181.87,431.08,676.31,708.47,144.3000,-151.68,-432.56


In [51]:
len(df_1), len(df_1.columns)

(5000, 12)

In [52]:
# File name to diagnostics mapping
df_diag = read_csv_from_s3(bucket_name, 'Chapman_Shaoxing/Diagnostics.csv', index_col=None)

In [53]:
df_diag.head()

,FileName,Rhythm,Beat,PatientAge,Gender,VentricularRate,AtrialRate,QRSDuration,QTInterval,QTCorrected,RAxis,TAxis,QRSCount,QOnset,QOffset,TOffset
0,MUSE_20180113_171327_27000,AFIB,RBBB TWC,85,MALE,117,234,114,356,496,81,-27,19,208,265,386
1,MUSE_20180112_073319_29000,SB,TWC,59,FEMALE,52,52,92,432,401,76,42,8,215,261,431
2,MUSE_20180111_165520_97000,SA,NONE,20,FEMALE,67,67,82,382,403,88,20,11,224,265,415
3,MUSE_20180113_121940_44000,SB,NONE,66,MALE,53,53,96,456,427,34,3,9,219,267,447
4,MUSE_20180112_122850_57000,AF,STDD STTC,73,FEMALE,162,162,114,252,413,68,-40,26,228,285,354


In [54]:
filenames = [Path(file).stem for file in cs_files]

In [55]:
# Build a mapping dictionary of filename - rhythm
mapping = dict(zip(df_diag['FileName'], df_diag['Rhythm']))

# Map filenames
diag_codes = pd.Series(filenames).map(mapping).values

In [56]:
np.unique(diag_codes, return_counts=True)

(array(['AF', 'AFIB', 'AT', 'AVNRT', 'AVRT', 'SA', 'SAAWR', 'SB', 'SR',
        'ST', 'SVT'], dtype=object),
 array([ 445, 1780,  121,   16,    8,  399,    7, 3889, 1826, 1568,  587]))

In [57]:
df_diag['Rhythm'].value_counts()

Rhythm
SB       3889
SR       1826
AFIB     1780
ST       1568
SVT       587
AF        445
SA        399
AT        121
AVNRT      16
AVRT        8
SAAWR       7
Name: count, dtype: int64

In [58]:
# Read all ecg data
ecg_data = []

for csfile in tqdm(cs_files):
    with fs.open(f"{bucket_name}/{csfile}", 'rb') as f:
        df_temp = pd.read_csv(f, header=None, index_col=None)
        ecg_data.append(df_temp.to_numpy())

100%|██████████| 10646/10646 [25:11<00:00,  7.04it/s]


In [59]:
ecg_data[0].shape

(5000, 12)

In [60]:
# Get all shapes to ensure all are (5000, 12)
ecg_len = []
nleads = []
for ecg in ecg_data:
    ecg_len.append(ecg.shape[0])
    nleads.append(ecg.shape[1])

print(np.unique(ecg_len, return_counts=True), np.unique(nleads, return_counts=True))

(array([1926, 5000]), array([    1, 10645])) (array([12]), array([10646]))


In [61]:
df_cs = pd.DataFrame({'filename': filenames,
                      'diag_code': diag_codes,
                      'ecg_signals': ecg_data})

In [62]:
# Numerical label mapping
# Labels
CODE_TO_LABEL = {code: label for label, code in enumerate(df_diag['Rhythm'].unique())}
print(CODE_TO_LABEL)

{'AFIB': 0, 'SB': 1, 'SA': 2, 'AF': 3, 'SR': 4, 'ST': 5, 'SVT': 6, 'AT': 7, 'AVNRT': 8, 'SAAWR': 9, 'AVRT': 10}


In [63]:
df_cs['label'] = df_cs['diag_code'].map(CODE_TO_LABEL)

In [64]:
df_cs['label'].value_counts(), df_cs['diag_code'].value_counts()

(label
 1     3889
 4     1826
 0     1780
 5     1568
 6      587
 3      445
 2      399
 7      121
 8       16
 10       8
 9        7
 Name: count, dtype: int64,
 diag_code
 SB       3889
 SR       1826
 AFIB     1780
 ST       1568
 SVT       587
 AF        445
 SA        399
 AT        121
 AVNRT      16
 AVRT        8
 SAAWR       7
 Name: count, dtype: int64)

In [65]:
# Save to s3
write_ecg_signals_to_s3(
    signals=ecg_data,
    labels=df_cs['label'],
    ids=filenames,
    dataset_name="cs",
    strat_fold=None
)

# remove from memory
del ecg_data
del df_cs

[cs] wrote 10,646 signals → s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/cs_signals.npz  (max_len=5000, lengths range=[1926, 5000])


In [66]:
# Check saved file
data = read_ecg_signals_from_s3(dataset_name="cs") 

[cs] loaded 10,646 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/cs_signals.npz


In [67]:
# Check labels
np.unique(data['labels'], return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
 array([1780, 3889,  399,  445, 1826, 1568,  587,  121,   16,    7,    8]))

In [68]:
# Check one ecg
data['signals'][0], data['ids'][0], data['labels'][0]

(array([[-165.33  , -358.97  , -121.71  , ...,   -7.8987, -315.16  ,
         -570.84  ],
        [-150.75  , -336.81  , -114.98  , ...,   35.389 , -269.51  ,
         -532.21  ],
        [-136.69  , -315.56  , -108.63  , ...,   76.572 , -225.83  ,
         -495.39  ],
        ...,
        [-171.04  ,   74.181 ,  172.08  , ...,  254.48  ,  129.34  ,
         -207.2   ],
        [-165.46  ,   79.57  ,  171.3   , ...,  259.51  ,  144.14  ,
         -190.64  ],
        [-160.69  ,   83.461 ,  169.81  , ...,  262.57  ,  157.62  ,
         -174.67  ]], shape=(5000, 12), dtype=float32),
 np.str_('MUSE_20180111_155115_19000'),
 np.int64(5))

In [69]:
# Free memory
del data